In [ ]:
# some util functions
import torch.nn.functional as F

def show_image(image,label,get_denormalize = True):

    image = image.permute(1,2,0)
    mean = torch.FloatTensor([0.485, 0.456, 0.406])
    std = torch.FloatTensor([0.229, 0.224, 0.225])

    if get_denormalize == True:
        image = image*std + mean
        image = np.clip(image,0,1)
        plt.imshow(image)
        plt.title(label)

    else:
        plt.imshow(image)
        plt.title(label)

def accuracy(y_pred,y_true):
    y_pred = F.softmax(y_pred,dim = 1)
    top_p,top_class = y_pred.topk(1,dim = 1)
    equals = top_class == y_true.view(*top_class.shape)
    return torch.mean(equals.type(torch.FloatTensor))





#Tumor MRI

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import helper

import torch
from tqdm import tqdm


#Download Dataset


In [ ]:
import kagglehub

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print(path)

In [ ]:
# inspect

for root, dirs, files in os.walk(path):
    print(root)
    break

# CONFIG

In [ ]:
DATA_DIR = '/kaggle/input/brain-tumor-mri-dataset'
TRAIN = 'Training'
VAL = 'Testing'

#model params
lr = 0.001
batch_size = 32
epochs = 2

model_name = 'tf_efficientnet_b8'
#model_namne = '"tf_efficientnet_b7.aa_in1k". #try whit one too.
#model_name = 'resnet34'


device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'using {device} for training')


#Load Datasets and Visualization of Samples

In [ ]:
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader

# Define transforms
transform = T.Compose([
    T.Resize((224, 224)),
    #T.RandomRotation(degrees = (-20,+20)),
    T.ToTensor(),
])

# Load datasets
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, TRAIN), transform=transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, VAL), transform=transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
del helper

In [ ]:
#from helper import show_image
class_name = train_dataset.classes
image, label = train_dataset[5000]

show_image(image, class_name[label], False)

plt.axis('off');

#Using UNET for Classification

*   List item
*   List item



In [ ]:
from torch import nn
from torchsummary import summary
import timm

In [ ]:
#load UNET and modify last layer for classifictaion

'''
model = timm.create_model(model_name, pretrained=True)

for param in model.parameters():
  param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(in_features=512, out_features=256, bias=True),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(in_features=256, out_features=4, bias=True)
)
model.fc
#summary(model, (3, 224, 224))
''';

#EfficientNet for classification

In [ ]:


# Load EfficientNet

model = timm.create_model(model_name, pretrained=True)

for param in model.parameters():
  param.requires_grad = False


#new classifier layer
model.classifier = nn.Sequential(
    nn.Linear(in_features = 2816, out_features = 625 ),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features = 625, out_features = 256),
    nn.ReLU(),
    nn.Linear(in_features = 256, out_features = 4)
)

#checking new classifier
model.classifier


In [ ]:
import timm
print([m for m in timm.list_models() if "efficientnet_b8" in m])

In [ ]:
class Trainer():

    def __init__(self, criterion, optimizer, scheduler):
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler


    def train_batch_loop(self, model, trainloader):
        train_acc = 0.0
        train_loss = 0.0

        for images, labels in tqdm(trainloader):
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)

            loss = self.criterion(logits,labels)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()
            train_acc += accuracy(logits,labels)

        return train_acc/len(trainloader), train_loss/len(trainloader)

    def valid_batch_loop(self,model, validloader):
        valid_acc = 0.0
        valid_loss = 0.0

        for images, labels in tqdm(validloader):
            images = images.to(device)
            labels = labels.to(device) # Fixed typo: 'lables' changed to 'labels'
            logits = model(images)

            loss = self.criterion(logits,labels)

            valid_loss += loss.item() # Changed to loss.item() for scalar accumulation
            valid_acc += accuracy(logits, labels)

        return valid_acc/len(validloader), valid_loss/len(validloader)


    def fit(self, model, trainloader, validloader, n_epochs):
        model = model.to(device)
        valid_min_loss = np.inf

        for epoch in range(n_epochs):
            model.train()
            avg_train_acc, avg_train_loss = self.train_batch_loop(model, trainloader)

            model.eval()
            avg_valid_acc, avg_valid_loss = self.valid_batch_loop(model, validloader)

            if avg_valid_loss < valid_min_loss:
                torch.save(model.state_dict(), 'model_EfficientNet.pt')
                print(f'saved best model with validation loss {avg_valid_loss}')
                valid_min_loss = avg_valid_loss

            print(f'epoch: {epoch}, Train Loss: {avg_train_loss}, Train Acc : {avg_train_acc} ')
            print(f'epoch: {epoch}, Valid Loss: {avg_valid_loss}, Train Acc : {avg_valid_acc} ')


In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import torch
import torch.nn as nn
from segmentation_models_pytorch.losses import DiceLoss

#criterion = DiceLoss('multiclass')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr =lr )
scheduler = None

trainer = Trainer(criterion, optimizer, scheduler)
trainer.fit(model, train_loader, test_loader, 2)